# Problem Set 4: Cross-Validation and Model Selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/05-04-template.ipynb)

You are advising a health insurer that wants to predict annual medical charges for planning purposes. Using the course copy of the insurance data at `data/insurance.csv`, you will explore predictive patterns, compare linear-regression feature sets, and use cross-validation and forward selection to choose among candidate models.

Each row is one insurance record. The outcome is `charges`, measured in dollars. The available predictors are `age`, `sex`, `bmi`, `children`, `smoker`, and `region`.

## Required work

Complete all five tasks in this notebook. Your submitted notebook must show:

1. the requested audit summaries and plots;
2. baseline and engineered-model training RMSE and mean five-fold CV-RMSE;
3. your additional engineered feature and rationale;
4. the complete forward-selection path and selected feature set; and
5. a three-model comparison with written interpretations.

Use the fixed assumptions stated in each task so answers are comparable. Write responses in your own words and keep all requested outputs visible. You may add or reorganize cells.

## How to complete and submit this notebook

1. Open the template using the course Google Colab link.
2. Before editing, select **File > Save a copy in Drive**. Work only in the saved copy and rename it so the filename includes `PS4` and your name.
3. Complete every required code and written-response task.
4. Select **Runtime > Run all** and confirm that every requested output is visible and no cell reports an error. Save the notebook after the run finishes.
5. Click **Share**. Under **General access**, choose **Anyone with the link**, set the role to **Viewer**, and copy the sharing link.
6. Submit the link to your completed Drive copy as the **Website URL** in Canvas. Do not submit the original GitHub template link.
7. After the deadline, do not edit the submitted notebook unless the instructor asks you to resubmit.

## Grading and use of LLMs

Please complete this problem set **without using an LLM to generate your code or written responses**.

You will receive full credit for a complete, good-faith submission. Accuracy is not the primary grading concern. The purpose of the assignment is to practice writing code, interpreting model-comparison results, and explaining your reasoning independently. Use the problem set to assess and deepen your understanding of the course material.

## 0. Setup

Run the supplied setup cell. Google Colab already includes these packages in a standard runtime. The instructor-provided helper first searches for a local `data/insurance.csv` and otherwise returns the public course copy on GitHub, so you do not need to upload the CSV or mount Google Drive.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score

plt.style.use("seaborn-v0_8-whitegrid")

PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


insurance_source = course_data_source("insurance.csv")

## Task 1 — Audit and explore the data

Load the course data with `pd.read_csv(insurance_source)`. Then produce all of the following visible outputs:

- the number of rows and columns;
- the missing-value count for every column;
- a numerical summary of `charges`;
- a clearly labeled histogram of `charges`;
- a smoker/non-smoker summary containing the number of records, mean charges, and median charges; and
- one additional clearly labeled plot that helps motivate a nonlinear transformation or an interaction from the available predictors.

Your plot may use `age`, `bmi`, `smoker`, or another predictor-based comparison. Do not create features from `charges`.

In [ ]:
# TODO
# Add or reorganize cells as needed.

**Your response:**

In 4–6 sentences, report the data dimensions and missingness, describe the shape and range of `charges`, and explain what the smoker comparison and your additional plot suggest for prediction. Refer to visible evidence.

## Task 2 — Define and score the baseline model

Create a baseline feature table using all six available predictors under these fixed rules:

- use `age`, `bmi`, and `children` as numeric features;
- use `no` as the omitted category for `smoker`;
- use `male` as the omitted category for `sex`;
- use `northeast` as the omitted category for `region`;
- do not include `charges` in the feature table; and
- fit `LinearRegression()`, which includes an intercept by default.

Display the encoded feature-column names and a short preview of the feature table.

**Hint**: When using `pd.get_dummies(..., drop_first=True)`, pay attention to the ordering of categories so that the specified reference categories are actually omitted.

Calculate and display:

- training RMSE from a model fitted and evaluated on all 1,338 records; and
- mean five-fold CV-RMSE using `KFold(n_splits=5, shuffle=True, random_state=505)`.

For the course measure, calculate RMSE separately in each fold and then average the five fold RMSE values. Use the same fold definition for every later model. Report RMSE in dollars.

In [ ]:
# TODO
# Add or reorganize cells as needed.

**Your response:**

Identify the reference category for each categorical predictor and explain why one category is represented by the zero/reference case when the model includes an intercept. Explain why training RMSE is usually optimistic relative to prediction error for new observations; the numeric results will be reported together in Task 5.

## Task 3 — Engineer and evaluate features

Create these three prescribed numeric features:

- `age_squared = age ** 2`;
- `bmi_squared = bmi ** 2`; and
- `bmi_smoker = bmi * smoker_yes`.

Also create one additional numeric feature of your own. It must be computed only from predictor information that would be available when making a prediction; it must not use `charges`. Give it a clear name and state its exact formula.

Fit an engineered-feature linear regression using every baseline encoded column, the three prescribed features, and your additional feature. Display a short preview of the four engineered columns, then display the model's training RMSE and mean CV-RMSE using the same five folds from Task 2.

In [ ]:
# TODO
# Add or reorganize cells as needed.

**Your response:**

Define your additional feature precisely and explain why it might help predict charges. State whether the visible mean CV-RMSE improves relative to the baseline; report the model errors together in Task 5.

## Task 4 — Run forward selection with CV-RMSE

Use the complete engineered feature pool from Task 3: every encoded baseline column, the three prescribed features, and your additional feature.

Implement forward selection under these fixed rules:

1. Begin with a mean-only model. Within each fold, it predicts the validation observations using the mean outcome from that fold's training observations.
2. At each step, try adding each remaining candidate feature column to the features already selected.
3. Score every candidate addition with the same five folds and mean fold RMSE definition used above.
4. Add the candidate with the lowest mean CV-RMSE. If scores are exactly tied, use the feature that appears first in your feature-pool order.
5. Continue until every candidate column has been added, even if a later step worsens CV-RMSE.
6. Select the recorded path row with the lowest mean CV-RMSE.

Show how the mean CV-RMSE changes as features are added. Show also the selected-feature collection, and the corresponding mean CV-RMSE.

In [ ]:
# TODO
# Add or reorganize cells as needed.

**Your response:**

Report the selected step, selected columns, omitted columns, and selected mean CV-RMSE. Describe at least two adjacent moves along the path and use their scores to explain whether every added feature improved mean CV-RMSE.

## Task 5 — Compare and interpret the three models

Compare the training RMSE and mean CV-RMSE for the following three models:

- baseline linear regression;
- engineered-feature linear regression; and
- forward-selection model.

In [ ]:
# TODO
# Add or reorganize cells as needed.

**Your response:**

In 6–9 sentences, answer all of the following:

- Which model has the lowest training RMSE? Report its value and explain why that result is expected when the full engineered model contains every candidate column used by the other two models.
- Which model has the lowest mean CV-RMSE? Report its value.
- How large is its CV-RMSE advantage over the next-best model, and is that difference practically large relative to the RMSE level?
- Does the forward path show that adding features always improves mean CV-RMSE?
- Why is CV-RMSE preferable to training RMSE for choosing a model intended for new observations?
- Why is the minimum score used during selection not an untouched final assessment, and how could you obtain a more honest final assessment?